## Redrob Hackathon - Reasoning Generation and Final Submission

### What this notebook does
This notebook turns the final top-100 shortlist into the exact submission CSV required by the challenge.

### Why this notebook is needed
The previous notebooks found and reranked strong candidates.
Now we need to:
- generate grounded reasoning
- assign final rank numbers
- format the score correctly
- export a valid CSV submission
- validate the file before upload

In [1]:
!pip install -q pandas numpy pyarrow

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
import json
import re
import subprocess
import sys

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
PROJECT_DIR = Path("/content/drive/MyDrive/Redrob_Hackathon")
DATA_DIR = PROJECT_DIR / "data"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

FINAL_SHORTLIST_PATH = ARTIFACTS_DIR / "final_shortlist_for_reasoning.parquet"
FEATURES_PATH = ARTIFACTS_DIR / "candidate_features.parquet"
JD_COMPASS_PATH = ARTIFACTS_DIR / "jd_compass.json"

In [5]:
print("Final shortlist exists:", FINAL_SHORTLIST_PATH.exists())
print("Features exists:", FEATURES_PATH.exists())
print("JD compass exists:", JD_COMPASS_PATH.exists())

Final shortlist exists: True
Features exists: True
JD compass exists: True


In [6]:
final_shortlist_df = pd.read_parquet(FINAL_SHORTLIST_PATH)
features_df = pd.read_parquet(FEATURES_PATH)

with open(JD_COMPASS_PATH, "r", encoding="utf-8") as f:
    jd_compass = json.load(f)

In [7]:
print("final_shortlist_df shape:", final_shortlist_df.shape)
print("features_df shape:", features_df.shape)
print("JD role:", jd_compass["jd_summary"]["role_title"])

final_shortlist_df shape: (100, 120)
features_df shape: (100000, 53)
JD role: Senior AI Engineer


Helper functions

In [20]:
def clean_text(value):
  if value is None:
    return ""
  value = str(value).strip()
  value = re.sub(r"\s+", " ", value)
  return value

In [8]:
def safe_float(value, default = 0.0):
   try:
       if value is None or value == "":
         return default
       return float(value)
   except Exception:
      return default

In [9]:
def pick_column(df, columns):
  for col in columns:
    if col in df.columns:
      return col
  raise  KeyError(f"None of these columns were found: {columns}" )

In [10]:
def unique_ordered(items):
    seen = set()
    result = []
    for item in items:
        item = clean_text(item)
        if item and item.lower() not in seen:
            seen.add(item.lower())
            result.append(item)
    return result

In [39]:
def count_sentences(text):
    text = clean_text(text)

    if not text:
        return 0

    # Count . only if it ends a real sentence:
    # followed by whitespace + uppercase letter, or end of string.
    sentence_endings = re.findall(
        r"[!?]+|(?<!\d)\.(?!\d)(?=\s+[A-Z]|$)",
        text
    )

    return len(sentence_endings) if sentence_endings else 1

In [12]:
def extract_matches(text, keywords, max_items=3):
    text = clean_text(text).lower()
    matches = []
    for kw in keywords:
        if kw.lower() in text:
            matches.append(kw)
    return unique_ordered(matches)[:max_items]

In [13]:
candidate_id_col = pick_column(final_shortlist_df, ["candidate_id"])
title_col = pick_column(final_shortlist_df, ["current_title_full", "current_title_retrieval", "current_title"])
company_col = pick_column(final_shortlist_df, ["current_company_full", "current_company_retrieval", "current_company"])
location_col = pick_column(final_shortlist_df, ["location_full", "location_retrieval", "location"])
years_col = pick_column(final_shortlist_df, ["years_of_experience_full", "years_of_experience_retrieval", "years_of_experience"])

profile_text_col = pick_column(final_shortlist_df, ["profile_text_full", "profile_text_retrieval", "profile_text"])
skill_text_col = pick_column(final_shortlist_df, ["skill_text_full", "skill_text_retrieval", "skill_text"])

score_col = pick_column(final_shortlist_df, ["final_rerank_score", "cross_encoder_score", "retrieval_stage_score"])

open_to_work_col = pick_column(final_shortlist_df, ["open_to_work_flag_full", "open_to_work_flag_retrieval", "open_to_work_flag"])
response_rate_col = pick_column(final_shortlist_df, ["recruiter_response_rate_full", "recruiter_response_rate_retrieval", "recruiter_response_rate"])
response_time_col = pick_column(final_shortlist_df, ["avg_response_time_hours_full", "avg_response_time_hours_retrieval", "avg_response_time_hours"])
notice_col = pick_column(final_shortlist_df, ["notice_period_days_full", "notice_period_days_retrieval", "notice_period_days"])
completeness_col = pick_column(final_shortlist_df, ["profile_completeness_score_full", "profile_completeness_score_retrieval", "profile_completeness_score"])
seniority_col = pick_column(final_shortlist_df, ["seniority_fit_score_full", "seniority_fit_score_retrieval", "seniority_fit_score"])
risk_col = pick_column(final_shortlist_df, ["total_risk_score_full", "total_risk_score_retrieval", "total_risk_score"])
availability_col = pick_column(final_shortlist_df, ["availability_strength_full", "availability_strength_retrieval", "availability_strength"])
location_fit_col = pick_column(final_shortlist_df, ["location_fit_flag_full", "location_fit_flag_retrieval", "location_fit_flag"])

In [14]:
print("Rows in shortlist:", len(final_shortlist_df))
print("Unique candidate ids:", final_shortlist_df[candidate_id_col].nunique())
print("Duplicate candidate ids:", final_shortlist_df[candidate_id_col].duplicated().sum())

assert len(final_shortlist_df) >= 100, "Need at least 100 candidates in the final shortlist."
assert final_shortlist_df[candidate_id_col].nunique() == len(final_shortlist_df), "Duplicate candidate IDs found."

display(
    final_shortlist_df[
        [
            candidate_id_col,
            title_col,
            years_col,
            score_col,
            risk_col
        ]
    ].head(10)
)

Rows in shortlist: 100
Unique candidate ids: 100
Duplicate candidate ids: 0


,candidate_id,current_title_full,years_of_experience_full,final_rerank_score,total_risk_score_full
0,CAND_0055905,Senior Machine Learning Engineer,8.1,78.016651,1
1,CAND_0005649,Senior Data Scientist,7.4,76.579591,0
2,CAND_0081846,Lead AI Engineer,6.7,74.986441,2
3,CAND_0086022,Senior Applied Scientist,5.3,72.848459,2
4,CAND_0069905,Applied ML Engineer,6.6,72.572914,0
5,CAND_0046525,Senior Machine Learning Engineer,6.1,71.782399,0
6,CAND_0077337,Staff Machine Learning Engineer,7.0,70.739877,2
7,CAND_0044883,AI Engineer,6.3,69.618009,0
8,CAND_0083307,Search Engineer,7.8,67.680995,3
9,CAND_0079387,AI Engineer,6.9,67.169423,2


In [15]:
core_keywords = jd_compass["skill_groups"]["core_ai_ml"]
retrieval_keywords = jd_compass["skill_groups"]["retrieval_ranking"]
production_keywords = jd_compass["skill_groups"]["production_engineering"]
evaluation_keywords = jd_compass["skill_groups"]["evaluation_and_quality"]


In [31]:
def build_reasoning(row):
    title = clean_text(row[title_col]) or "Candidate"
    company = clean_text(row[company_col])
    location = clean_text(row[location_col])
    years = safe_float(row[years_col])

    profile_text = clean_text(row[profile_text_col])
    skill_text = clean_text(row[skill_text_col])
    combined_text = f"{profile_text} {skill_text}"

    matched_core = extract_matches(combined_text, core_keywords, max_items=3)
    matched_retrieval = extract_matches(combined_text, retrieval_keywords, max_items=3)
    matched_production = extract_matches(combined_text, production_keywords, max_items=3)
    matched_evaluation = extract_matches(combined_text, evaluation_keywords, max_items=2)

    open_to_work = bool(int(safe_float(row[open_to_work_col], 0)))
    response_rate = safe_float(row[response_rate_col], 0.0)
    response_time = safe_float(row[response_time_col], 0.0)
    notice_period = safe_float(row[notice_col], 0.0)
    completeness = safe_float(row[completeness_col], 0.0)
    seniority = safe_float(row[seniority_col], 0.0)
    risk_score = safe_float(row[risk_col], 0.0)
    location_fit = int(safe_float(row[location_fit_col], 0.0))

    positive_parts = []
    if matched_retrieval:
        positive_parts.append("retrieval/ranking experience around " + ", ".join(matched_retrieval))
    if matched_production:
        positive_parts.append("production evidence with " + ", ".join(matched_production))
    if matched_evaluation:
        positive_parts.append("evaluation exposure through " + ", ".join(matched_evaluation))
    if matched_core:
        positive_parts.append("core AI/ML background with " + ", ".join(matched_core))

    if years >= 5 and years <= 9 and (matched_retrieval or matched_production):
        sentence1 = (
            f"{title} with {years:.1f} years looks like a strong fit for the Senior AI Engineer role"
            f"{' at ' + company if company else ''}"
            f"{' from ' + location if location else ''}, with "
            + "; ".join(positive_parts[:2])
            + "."
        )
    elif years > 12 and (matched_retrieval or matched_production):
        sentence1 = (
            f"{title} with {years:.1f} years is a bit above the ideal 5 to 9 year band, "
            f"but still relevant because of "
            + "; ".join(positive_parts[:2])
            + "."
        )
    elif years < 5 and (matched_retrieval or matched_production):
        evidence_text = "; ".join(positive_parts[:2])

        sentence1 = (
            f"{title} with {years:.1f} years is slightly junior for the target seniority band, "
            f"but shows relevant evidence through {evidence_text}."
    )
    else:
        sentence1 = (
            f"{title} with {years:.1f} years has some useful signals, "
            f"but the evidence is lighter than the strongest shortlisted profiles."
        )

    concerns = []
    if not open_to_work:
        concerns.append("the candidate is not marked open to work")
    if response_rate < 0.25:
        concerns.append(f"recruiter response rate is only {response_rate:.2f}")
    if response_time > 150:
        concerns.append(f"response time is {response_time:.0f} hours")
    if notice_period > 120:
        concerns.append(f"notice period is {notice_period:.0f} days")
    if completeness < 45:
        concerns.append(f"profile completeness is {completeness:.1f}")
    if risk_score >= 4:
        concerns.append("the profile carries stronger trust risk than cleaner fits")
    if location_fit == 0:
        concerns.append("location is not an obvious match for the preferred cities")

    if open_to_work and response_rate >= 0.5 and notice_period <= 90 and risk_score < 4:
        sentence2 = (
            f"Availability signals look workable, with open-to-work status, "
            f"{response_rate:.2f} recruiter response rate, and {notice_period:.0f}-day notice period."
        )
    elif concerns:
        sentence2 = "The main concern is " + "; ".join(concerns[:2]) + "."
    else:
        sentence2 = "The profile looks usable, with no major red flags from the observed signals."

    reasoning = sentence1.strip()
    reasoning = reasoning + " " + sentence2.strip()
    return reasoning.strip()

In [32]:
final_shortlist_df = final_shortlist_df.copy()
final_shortlist_df["reasoning"] = final_shortlist_df.apply(build_reasoning, axis=1)

display(
    final_shortlist_df[
        [
            candidate_id_col,
            title_col,
            years_col,
            score_col,
            "reasoning"
        ]
    ].head(10)
)

,candidate_id,current_title_full,years_of_experience_full,final_rerank_score,reasoning
0,CAND_0055905,Senior Machine Learning Engineer,8.1,78.016651,Senior Machine Learning Engineer with 8.1 year...
1,CAND_0005649,Senior Data Scientist,7.4,76.579591,Senior Data Scientist with 7.4 years looks lik...
2,CAND_0081846,Lead AI Engineer,6.7,74.986441,Lead AI Engineer with 6.7 years looks like a s...
3,CAND_0086022,Senior Applied Scientist,5.3,72.848459,Senior Applied Scientist with 5.3 years looks ...
4,CAND_0069905,Applied ML Engineer,6.6,72.572914,Applied ML Engineer with 6.6 years looks like ...
5,CAND_0046525,Senior Machine Learning Engineer,6.1,71.782399,Senior Machine Learning Engineer with 6.1 year...
6,CAND_0077337,Staff Machine Learning Engineer,7.0,70.739877,Staff Machine Learning Engineer with 7.0 years...
7,CAND_0044883,AI Engineer,6.3,69.618009,AI Engineer with 6.3 years looks like a strong...
8,CAND_0083307,Search Engineer,7.8,67.680995,Search Engineer with 7.8 years looks like a st...
9,CAND_0079387,AI Engineer,6.9,67.169423,AI Engineer with 6.9 years looks like a strong...


In [40]:
TEAM_ID = "team_hehehe"

submission_df = final_shortlist_df[[candidate_id_col, score_col, "reasoning"]].copy()

submission_df = submission_df.sort_values(
    by=[score_col, candidate_id_col],
    ascending=[False, True]
).reset_index(drop=True)

submission_df["rank"] = np.arange(1, len(submission_df) + 1)

scores = submission_df[score_col].astype(float).to_numpy()

if scores.max() > scores.min():
    submission_df["score"] = (
        (scores - scores.min()) / (scores.max() - scores.min())
    )
else:
    submission_df["score"] = np.ones_like(scores)

submission_df = submission_df[
    [candidate_id_col, "rank", "score", "reasoning"]
].copy()

submission_df.columns = [
    "candidate_id",
    "rank",
    "score",
    "reasoning"
]

display(submission_df.head(10))

,candidate_id,rank,score,reasoning
0,CAND_0055905,1,1.000000,Senior Machine Learning Engineer with 8.1 year...
1,CAND_0005649,2,0.952757,Senior Data Scientist with 7.4 years looks lik...
2,CAND_0081846,3,0.900383,Lead AI Engineer with 6.7 years looks like a s...
3,CAND_0086022,4,0.830098,Senior Applied Scientist with 5.3 years looks ...
4,CAND_0069905,5,0.821040,Applied ML Engineer with 6.6 years looks like ...
5,CAND_0046525,6,0.795052,Senior Machine Learning Engineer with 6.1 year...
6,CAND_0077337,7,0.760780,Staff Machine Learning Engineer with 7.0 years...
7,CAND_0044883,8,0.723899,AI Engineer with 6.3 years looks like a strong...
8,CAND_0083307,9,0.660220,Search Engineer with 7.8 years looks like a st...
9,CAND_0079387,10,0.643403,AI Engineer with 6.9 years looks like a strong...


In [41]:
submission_df["reasoning_sentence_count"] = submission_df["reasoning"].apply(
    count_sentences
)

print(submission_df["reasoning_sentence_count"].value_counts().sort_index())

display(
    submission_df.loc[
        submission_df["reasoning_sentence_count"] > 2,
        ["candidate_id", "reasoning", "reasoning_sentence_count"]
    ]
)

reasoning_sentence_count
1     4
2    96
Name: count, dtype: int64


,candidate_id,reasoning,reasoning_sentence_count


In [43]:
print("Final submission rows:", len(submission_df))
print("Unique candidate ids:", submission_df["candidate_id"].nunique())
print("Duplicate candidate ids:", submission_df["candidate_id"].duplicated().sum())
print("Ranks:", submission_df["rank"].min(), "to", submission_df["rank"].max())

assert len(submission_df) == 100, "Submission must have exactly 100 rows."
assert submission_df["candidate_id"].nunique() == 100, "Candidate IDs must be unique."
assert list(submission_df["rank"]) == list(range(1, 101)), "Ranks must be 1 through 100."
assert submission_df["score"].is_monotonic_decreasing, "Scores must be non-increasing."

Final submission rows: 100
Unique candidate ids: 100
Duplicate candidate ids: 0
Ranks: 1 to 100


In [51]:
submission_df["reasoning_sentence_count"] = submission_df["reasoning"].apply(count_sentences)
print(submission_df["reasoning_sentence_count"].value_counts().sort_index())

assert submission_df["reasoning"].str.strip().ne("").all(), "Reasoning cannot be empty."
assert submission_df["reasoning_sentence_count"].between(1, 2).all(), "Reasoning must be 1 to 2 sentences."

display(submission_df[["candidate_id", "rank", "score", "reasoning"]].head(10))

reasoning_sentence_count
1     4
2    96
Name: count, dtype: int64


,candidate_id,rank,score,reasoning
0,CAND_0055905,1,1.000000,Senior Machine Learning Engineer with 8.1 year...
1,CAND_0005649,2,0.952757,Senior Data Scientist with 7.4 years looks lik...
2,CAND_0081846,3,0.900383,Lead AI Engineer with 6.7 years looks like a s...
3,CAND_0086022,4,0.830098,Senior Applied Scientist with 5.3 years looks ...
4,CAND_0069905,5,0.821040,Applied ML Engineer with 6.6 years looks like ...
5,CAND_0046525,6,0.795052,Senior Machine Learning Engineer with 6.1 year...
6,CAND_0077337,7,0.760780,Staff Machine Learning Engineer with 7.0 years...
7,CAND_0044883,8,0.723899,AI Engineer with 6.3 years looks like a strong...
8,CAND_0083307,9,0.660220,Search Engineer with 7.8 years looks like a st...
9,CAND_0079387,10,0.643403,AI Engineer with 6.9 years looks like a strong...


In [52]:
final_submission_df = submission_df[
    ["candidate_id", "rank", "score", "reasoning"]
].copy()

In [53]:
submission_path = OUTPUTS_DIR / f"{TEAM_ID}.csv"

final_submission_df.to_csv(
    submission_path,
    index=False,
    encoding="utf-8"
)

print("Saved clean submission to:", submission_path)

Saved clean submission to: /content/drive/MyDrive/Redrob_Hackathon/outputs/team_hehehe.csv


In [55]:
print("\nFinal CSV columns:")
print(final_submission_df.columns.tolist())

final_submission_df.head()


Final CSV columns:
['candidate_id', 'rank', 'score', 'reasoning']


,candidate_id,rank,score,reasoning
0,CAND_0055905,1,1.000000,Senior Machine Learning Engineer with 8.1 year...
1,CAND_0005649,2,0.952757,Senior Data Scientist with 7.4 years looks lik...
2,CAND_0081846,3,0.900383,Lead AI Engineer with 6.7 years looks like a s...
3,CAND_0086022,4,0.830098,Senior Applied Scientist with 5.3 years looks ...
4,CAND_0069905,5,0.821040,Applied ML Engineer with 6.6 years looks like ...


In [56]:
submission_summary = {
    "rows": int(len(submission_df)),
    "candidate_ids_unique": bool(submission_df["candidate_id"].nunique() == 100),
    "rank_min": int(submission_df["rank"].min()),
    "rank_max": int(submission_df["rank"].max()),
    "score_min": float(submission_df["score"].min()),
    "score_max": float(submission_df["score"].max()),
    "output_file": str(submission_path),
}

with open(OUTPUTS_DIR / "submission_summary.json", "w", encoding="utf-8") as f:
    json.dump(submission_summary, f, indent=4)

print(submission_summary)

{'rows': 100, 'candidate_ids_unique': True, 'rank_min': 1, 'rank_max': 100, 'score_min': 0.0, 'score_max': 1.0, 'output_file': '/content/drive/MyDrive/Redrob_Hackathon/outputs/team_hehehe.csv'}


In [57]:
validator_candidates = [
    PROJECT_DIR / "validate_submission.py",
    DATA_DIR / "validate_submission.py"
]

validator_path = None
for path in validator_candidates:
    if path.exists():
        validator_path = path
        break

if validator_path is None:
    print("Validator script not found in the expected locations.")
else:
    print("Using validator:", validator_path)

    result = subprocess.run(
        [sys.executable, str(validator_path), str(submission_path)],
        capture_output=True,
        text=True
    )

    print("STDOUT:\n", result.stdout)
    print("STDERR:\n", result.stderr)

Using validator: /content/drive/MyDrive/Redrob_Hackathon/data/validate_submission.py
STDOUT:
 Submission is valid.

STDERR:
 


In [58]:
display(submission_df[["candidate_id", "rank", "score", "reasoning"]].head(20))

,candidate_id,rank,score,reasoning
0,CAND_0055905,1,1.000000,Senior Machine Learning Engineer with 8.1 year...
1,CAND_0005649,2,0.952757,Senior Data Scientist with 7.4 years looks lik...
2,CAND_0081846,3,0.900383,Lead AI Engineer with 6.7 years looks like a s...
3,CAND_0086022,4,0.830098,Senior Applied Scientist with 5.3 years looks ...
4,CAND_0069905,5,0.821040,Applied ML Engineer with 6.6 years looks like ...
5,CAND_0046525,6,0.795052,Senior Machine Learning Engineer with 6.1 year...
6,CAND_0077337,7,0.760780,Staff Machine Learning Engineer with 7.0 years...
7,CAND_0044883,8,0.723899,AI Engineer with 6.3 years looks like a strong...
8,CAND_0083307,9,0.660220,Search Engineer with 7.8 years looks like a st...
9,CAND_0079387,10,0.643403,AI Engineer with 6.9 years looks like a strong...
